# Day 1 — Docker for AI Apps

---

Today, in 75 minutes:

1. What Docker is and why every AI app in 2026 ships in a container
2. Write a `Dockerfile` for a FastAPI + LLM app
3. Build, run, and inspect the image
4. Two AI-specific Docker rules that will save you hours


## 1. What is Docker?

Docker packages your app **plus its OS + Python + dependencies** into a single portable unit called an **image**. You run instances of that image (called **containers**) anywhere Docker is installed — your laptop, a cloud VM, a Kubernetes cluster.

**Why AI apps love Docker:**
- Reproducible Python + CUDA + model-file versions (huge for AI where "works on my machine" bites hardest)
- Cloud PaaS platforms (Render, Fly, Railway) all take a Dockerfile as input
- One image → laptop, staging, production


## 2. The three things in a Dockerfile

Every Dockerfile follows the same shape:

1. **FROM** — pick a base image (a Linux + Python + friends)
2. **COPY / RUN** — install deps and copy your code
3. **CMD** — the command to run when the container starts


## 3. A minimal Dockerfile for a FastAPI app

```dockerfile
FROM python:3.11-slim

WORKDIR /app

# Install deps FIRST so Docker caches this layer
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Then copy the app code
COPY . .

# The port your app listens on
EXPOSE 8000

# Start the server
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

**Why `COPY requirements.txt` before `COPY . .`?** Docker caches each layer. If you edit `main.py`, only the `COPY . .` layer rebuilds — the slow `pip install` reuses cache. This one trick saves minutes on every rebuild.


## 4. Build and run


In [ ]:
# In a terminal (not in Jupyter):
!docker build -t my-rag-app .
!docker run --rm -p 8000:8000 --env-file .env my-rag-app


**Understand the run flags:**
- `-p 8000:8000` — map host port 8000 to container port 8000
- `--env-file .env` — pass your API keys as env vars
- `--rm` — auto-clean the container when it exits

Open http://localhost:8000/docs — the app runs, just like it did with `uvicorn` locally.


## 5. Two AI-specific Docker rules

### Rule 1 — Don't bake giant model files into the image
A 6 GB model file in your image makes deploys slow and expensive. Options:

- **Download at container start** (`RUN` after startup, cached on disk)
- **Volume-mount from a shared model directory**
- **Serve models from a separate service** (Ollama, Together, Hugging Face)

Rule of thumb: **containers should stay under 1 GB.**

### Rule 2 — Use `.dockerignore`

Create a `.dockerignore` next to your Dockerfile:

```
.git
.venv
__pycache__
*.pyc
.env
data/
notebooks/
```

Without this, `COPY . .` will happily bake in your virtualenv (500 MB) and secrets (`.env`).


## 6. Multi-stage builds (optional — smaller images)

For a truly small image, use a **multi-stage** build: install deps in one stage, copy just the built artifacts to a slim runtime stage.

```dockerfile
FROM python:3.11 AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

FROM python:3.11-slim
WORKDIR /app
COPY --from=builder /root/.local /root/.local
ENV PATH=/root/.local/bin:$PATH
COPY . .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

For freshers: **single-stage is fine to start.** Do multi-stage when your image gets big or you care about cold-start latency.


## 7. Inspecting a running container

- `docker ps` — list running containers
- `docker logs <container>` — see stdout/stderr
- `docker exec -it <container> bash` — open a shell inside
- `docker image ls` — see image sizes on your machine
- `docker system prune` — reclaim disk (careful, it deletes stopped stuff)


## Recap

- **Docker** = your app + OS + deps in one portable image.
- Dockerfile essentials: `FROM` + `RUN pip install` + `COPY` + `CMD`.
- **`COPY requirements.txt` before `COPY . .`** for caching.
- **Keep images small**: `.dockerignore` and don't bake in giant model files.
- Run with `-p 8000:8000 --env-file .env`.
- **Next class:** GitHub Actions — automate testing and building on every push.
